In [1]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### 8-1 Introduccion

##### 8-2 Cuando usar SQL

##### 8-3 Cuando usar ETL

##### 8-4 Cuando usar otras herramientas

##### 8.5 Comentar código

##### 8.6 Estilo

##### 8.7 Almacenamiento de código

##### 8-8 Orden de Evaluacion


Se ejecutan en el siguiente orden:

- FROM (incluyendo JOINS en su clausula ON)

- WHERE

- GROUP BY

- HAVING

- WINDOW FUNCTION

- SELECT

- DISTINCT

- UNION

- ORDER BY

- LIMIT y OFFSET

##### 8-9 Tablas Temporales

Las tablas temporales son tablas que pueden ser creadas y existiran unicamente durante la sesión actual y se eliminaran automaticamente cuando la sesión finalice.

Para crear una tabla temporal se requiere permiso de escritura en la base de datos 


In [2]:
# Crear un esquema (borralo antes si ya existe)

query1 = r"""
DROP DATABASE IF EXISTS new_schema;
"""

query2 = r"""
CREATE DATABASE new_schema;
"""

with engine.connect() as connection:
    connection.execute(text(query1))    
    connection.execute(text(query2))
    connection.commit()


In [ ]:
# Crear una tabla temporal */

query1 = r"""
CREATE TEMPORARY TABLE new_schema.T_ciudades(
ciudad_id	INT PRIMARY KEY AUTO_INCREMENT,
ciudad		VARCHAR(25) 		
);
"""

with engine.begin() as conn:
    conn.execute(text(query1)) #Ejecutamos cada consulta dentro de la conexión. Esto asegura que ambas consultas se ejecuten en el mismo contexto de transacción.
    connection.commit()


In [4]:
# Si es necesario podemos eliminar la tabla temporal

#query = r"""
#DROP TEMPORARY TABLE new_schema.T_ciudades;
#"""
#with engine.begin() as conn:
#    conn.execute(text(query)) #Ejecutamos cada consulta dentro de la conexión. Esto asegura que ambas consultas se ejecuten en el mismo contexto de transacción.
#    connection.commit()

In [5]:
# Visualizar la tabla temporal

query = r"""
select
    * 
from
    new_schema.T_ciudades;
"""
df = pd.read_sql_query(query, engine)
df.head()

,ciudad_id,ciudad


In [6]:
# Insertar filas a la tabla temporal */

query = r"""
INSERT INTO new_schema.T_ciudades(ciudad)
SELECT 	distinct ciudad 
FROM 	cohort.supertienda;
"""

with engine.begin() as conn:
    conn.execute(text(query)) #Ejecutamos cada consulta dentro de la conexión. Esto asegura que ambas consultas se ejecuten en el mismo contexto de transacción.
    connection.commit()

In [7]:
# Visualizar la tabla temporal

query = r"""
select
    * 
from
    new_schema.T_ciudades;
"""
df = pd.read_sql_query(query, engine)
df.head()

,ciudad_id,ciudad
0,1,Houston
1,2,Naperville
2,3,Philadelphia
3,4,Henderson
4,5,Athens


In [8]:
# Se puede crear y poblar la tabla temporal en una sola sentencia */

# Borrar la tabla temporal si ya existe
query1 = r"""
DROP TABLE new_schema.T_ciudades;
"""

# Crear la tabla temporal a partir de una consulta
query2 = r"""
CREATE TEMPORARY TABLE new_schema.T_ciudades AS
SELECT distinct ciudad
FROM cohort.supertienda;
"""
with engine.begin() as conn:
    conn.execute(text(query1))
    conn.execute(text(query2))
    connection.commit()

In [9]:
# Visualizar la tabla temporal

query = r"""
select
    * 
from
    new_schema.T_ciudades;
"""
df = pd.read_sql_query(query, engine)
df.head()

,ciudad
0,Houston
1,Naperville
2,Philadelphia
3,Henderson
4,Athens


##### 8-10 CTE: Expresiones de Tabla Comunes

Una CTE (Common Table Expression o Expresión de Tabla Común) es un conjunto de resultados temporales con nombre que se define al inicio de una consulta usando la cláusula WITH.

Funciona como una tabla virtual o vista que solo existe mientras se está ejecutando la instrucción principal (SELECT, INSERT, UPDATE o DELETE) y no se guarda de forma permanente en la base de datos. 

Están disponibles en MySQL a partir de la versión 8.0.

In [10]:
# Veamos la tabla supertienda

query = r"""
select
    *
from
    cohort.supertienda;
"""
df = pd.read_sql_query(text(query), engine)
df.head(5)

,venta_id,Categoria,Ciudad,Pais/Región,Nombre_del_Cliente,Proveedor,Fecha_de_Orden,ID_Orden,Codigo_Postal,Nombre_de_Producto,Region,Segmento,Fecha_de_envio,Estado/Provincia,Descuento,Ganancia,Cantidad,Ventas
0,1,Suministros de Oficina,Houston,United States,Darren Powers,Message Book,2019-01-03,US-2019-103800,77095,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",Central,Consumidor,2019-01-07,Texas,0.2,5.5512,2,16.448
1,2,Suministros de Oficina,Naperville,United States,Phillina Ober,GBC,2019-01-04,US-2019-112326,60540,GBC Standard Plastic Binding Systems Combs,Central,Home Office,2019-01-08,Illinois,0.8,-5.4870,2,3.540
2,3,Suministros de Oficina,Naperville,United States,Phillina Ober,Avery,2019-01-04,US-2019-112326,60540,Avery 508,Central,Home Office,2019-01-08,Illinois,0.2,4.2717,3,11.784
3,4,Suministros de Oficina,Naperville,United States,Phillina Ober,SAFCO,2019-01-04,US-2019-112326,60540,SAFCO Boltless Steel Shelving,Central,Home Office,2019-01-08,Illinois,0.2,-64.7748,3,272.736
4,5,Suministros de Oficina,Philadelphia,United States,Mick Brown,Avery,2019-01-05,US-2019-141817,19143,Avery Hi-Liter EverBold Pen Style Fluorescent ...,Este,Consumidor,2019-01-12,Pennsylvania,0.2,4.8840,3,19.536


In [11]:
# Usar un CTE (Common Table Expression) para calcular la cantidad de clientes que vuelven a comprar k años después de su primera compra

query = r"""
WITH cte_primer_compra as(
SELECT
    nombre_del_cliente,
    MIN(fecha_de_orden) AS primer_compra
FROM
    cohort.supertienda
GROUP BY
    1
)
SELECT
    timestampdiff(year, a.primer_compra, b.fecha_de_orden) as periodos,
    COUNT(distinct(b.nombre_del_cliente)) as cohorte_retenida
FROM
    cte_primer_compra AS a JOIN cohort.supertienda AS b ON a.nombre_del_cliente = b.nombre_del_cliente
GROUP BY 
    1
;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,periodos,cohorte_retenida
0,0,800
1,1,619
2,2,588
3,3,403


##### 8.11 Grouping sets

In [12]:
# Visualizar la tabla supermarket_sales

query = r"""
SELECT 
    *
FROM
    crearsetscomplejos.supermarket_sales
"""
df = pd.read_sql_query(text(query), engine)
df.head(5)

,supermarket_sales_id,Invoice_ID,Branch,City,Customer_type,Gender,Product_line,Unit_price,Quantity,Tax_5%,Total,Date,Time,Payment,cogs,gross_margin_percentage,gross_income,Rating
0,1,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,01/05/2019 12:00:00 a. m.,0 days 13:08:00,Ewallet,522.83,4.761905,26.1415,9.1
1,2,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,03/08/2019 12:00:00 a. m.,0 days 10:29:00,Cash,76.40,4.761905,3.8200,9.6
2,3,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,03/03/2019 12:00:00 a. m.,0 days 13:23:00,Credit card,324.31,4.761905,16.2155,7.4
3,4,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,0 days 20:33:00,Ewallet,465.76,4.761905,23.2880,8.4
4,5,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,02/08/2019 12:00:00 a. m.,0 days 10:37:00,Ewallet,604.17,4.761905,30.2085,5.3


In [13]:
# Obtener el total de ingreso por tipo de cliente

query = r"""
SELECT
    customer_type,
    ROUND(SUM(total),2) AS venta_total
FROM
    crearsetscomplejos.supermarket_sales
GROUP BY
    1;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,customer_type,venta_total
0,Member,164223.44
1,Normal,158743.31


In [14]:
# Agregar a la tabla anterior los totales de todo */

query = r"""
SELECT
    customer_type,
    ROUND(SUM(total),2) AS venta_total
FROM
    crearsetscomplejos.supermarket_sales
GROUP BY
    1 WITH ROLLUP;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,customer_type,venta_total
0,Member,164223.44
1,Normal,158743.31
2,NaN,322966.75


In [19]:
# Es posible usar la sentencia WITH ROLLUP con multiples columnas de agrupamiento */

query = r"""
SELECT
    customer_type,
    gender,
    branch,
    round(sum(total),2) as venta_total
FROM
    crearsetscomplejos.supermarket_sales
GROUP BY
    1,2,3 WITH ROLLUP
ORDER BY
    1,2,3;
"""
df = pd.read_sql_query(text(query), engine)
df.head(100)


,customer_type,gender,branch,venta_total
0,NaN,NaN,NaN,322966.75
1,Member,NaN,NaN,164223.44
2,Member,Female,NaN,88146.94
3,Member,Female,A,26643.07
4,Member,Female,B,26850.43
5,Member,Female,C,34653.44
6,Member,Male,NaN,76076.50
7,Member,Male,A,26994.41
8,Member,Male,B,26854.25
9,Member,Male,C,22227.84


##### 8-12 Muestreo

In [21]:
# En MySQL, la función MOD() devuelve el residuo de la división de un número por otro.

query = r"""
select mod(17786,100) as residuo;
"""
df = pd.read_sql(text(query),engine)
df.head()

,residuo
0,86


In [24]:
# Hacer una consulta para obtener solo el 1 porciento de los datos */

query = r"""
SELECT
    storms_id,
    name
FROM
    anomalias.storms
WHERE
    mod(storms_id,100)=86;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,storms_id,name
0,86,AL051881
1,186,AL111909
2,286,AL101926
3,386,AL131936
4,486,AL051945
5,586,AL021949
6,686,Charlie
7,786,Gail
8,886,Edith
9,986,Audrey


##### 8-13 Reduccion de Dimensionalidad

In [27]:
##### Obtener una lista de paises que mas reseñas hacen sobre los parques de Disney */

query = r"""
SELECT
    pais_origen,
    count(*) as reseñas
FROM
    analisistexto.disneyclean
GROUP BY
    1
ORDER BY
    2 desc;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,pais_origen,reseñas
0,US,14551
1,UK,9751
2,Australia,4679
3,Canada,2235
4,India,1511
5,Philippines,1070
6,Singapore,1037
7,New Zealand,756
8,Malaysia,588
9,Hong Kong,554


In [38]:
# Función RANK()
# La función RANK() en MySQL es una función de ventana que asigna una posición o un rango numérico a cada fila de un conjunto de resultados, 
# basándose en un orden especificado mediante la cláusula ORDER BY
# Usando la consulta anterior, agregar una columna para rankear a los paises segun su numero de reseñas */

query = r"""
SELECT
    pais_origen,
    COUNT(*) AS reseñas,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS ranking
FROM
    analisistexto.disneyclean
GROUP BY
    1
ORDER BY
    2 DESC;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,pais_origen,reseñas,ranking
0,US,14551,1
1,UK,9751,2
2,Australia,4679,3
3,Canada,2235,4
4,India,1511,5
5,Philippines,1070,6
6,Singapore,1037,7
7,New Zealand,756,8
8,Malaysia,588,9
9,Hong Kong,554,10


In [43]:
# Usando la consulta anterior obtener los 5 primero paises con mas reseñas y los demas agruparlos en una categoria llamada "otros" */

query = r"""
SELECT
    pais,
    SUM(reseñas) AS suma_reseñas
FROM
    (
    SELECT
        pais_origen,
        COUNT(*) AS reseñas,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS ranking,
        CASE
            WHEN RANK() OVER (ORDER BY COUNT(*) DESC) <= 5 THEN pais_origen
            ELSE "Otros"
        END as pais
    FROM
        analisistexto.disneyclean
    GROUP BY
        1
    ORDER BY
        2 DESC
    ) as z
GROUP BY
    1;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,pais,suma_reseñas
0,US,14551.0
1,UK,9751.0
2,Australia,4679.0
3,Canada,2235.0
4,India,1511.0
5,Otros,9929.0


##### 8 -14 Datos personales

In [46]:
# De la tabla supertienda obtener el total de venta por cada cliente pero omitir los datos personales del cliente.

query = r"""
SELECT
    ROW_NUMBER() OVER (ORDER BY nombre_del_cliente) AS cliente,
    venta_total
FROM
    (
    SELECT
        nombre_del_cliente,
        ROUND(SUM(ventas),2) AS venta_total
    FROM
        cohort.supertienda
    GROUP BY
        1
    ORDER BY
        1
    ) AS z;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,cliente,venta_total
0,1,886.16
1,2,1744.70
2,3,3050.69
3,4,7755.62
4,5,3250.34
5,6,3255.31
6,7,14473.57
7,8,1735.51
8,9,58.82
9,10,966.71


In [ ]:
# En MySQL, la función MD5() calcula un hash MD5 de una cadena de texto dada. Este hash es una representación hexadecimal de 32 caracteres 
# de la cadena original, y se utiliza comúnmente para verificar la integridad de datos o como una forma de almacenar contraseñas de forma 
# segura (aunque no es la forma más segura actualmente).

query = r"""
SELECT
    md5(nombre_del_cliente) AS cliente,
    venta_total
FROM
    (
    SELECT
        nombre_del_cliente,
        ROUND(sum(ventas),2) as venta_total
    FROM
        cohort.supertienda
    GROUP BY 1
    ORDER BY 1
) AS z;
"""
df = pd.read_sql(text(query), engine)
df.head(10)